# Deep Past Challenge — ByT5-Large Full Scratch Training v1

### Strategy (from competition discussions)
- **3-Phase Curriculum**: Broad → Domain → Polish
- **V3 Preprocessing**: Match test data normalization exactly (ḫ→h, gaps unified, fractions→Unicode, PN→gap)
- **All Data Sources**: ~14,700 pairs across strategies + ORACC + lexicon
- **Key insight**: "95% preprocessing effort" (Jack, 19th place) — preprocessing > model architecture

### Kaggle Setup
- **GPU**: H100 (preferred) or A100
- **Internet**: ON (first run to download google/byt5-large)
- **Input datasets**: `akkadian-processed-data` (train_split.csv, val_split.csv)
- **Expected time**: ~4-6 hours on H100

### Phases
1. **Phase 1**: All data (14K+ pairs) — learn Akkadian→English broadly
2. **Phase 2**: OA sentence pairs only (~9.8K) — domain specialization
3. **Phase 3**: Highest quality pairs (~1.1K strategy_a) — match test distribution

In [ ]:
!pip install -q sacrebleu transformers[torch] accelerate

import os, json, math, gc, time, shutil, re, unicodedata, copy
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from torch.utils.data import Dataset
import sacrebleu

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f}GB")
    IS_H100 = 'H100' in gpu_name
    IS_A100 = 'A100' in gpu_name
else:
    IS_H100 = IS_A100 = False
    print("WARNING: No GPU detected!")

start_time = time.time()

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# --- Data Paths ---
TRAIN_FILE = "/kaggle/input/akkadian-processed-data/train_split.csv"
VAL_FILE = "/kaggle/input/akkadian-processed-data/val_split.csv"
if not Path(TRAIN_FILE).exists():
    TRAIN_FILE = "../data_processed/combined/train_split.csv"
    VAL_FILE = "../data_processed/combined/val_split.csv"

# --- Model ---
# TRAIN_FROM_SCRATCH = True  -> Full training from google/byt5-large (target 40+)
# TRAIN_FROM_SCRATCH = False -> Conservative fine-tuning from pre-trained
TRAIN_FROM_SCRATCH = True
BASE_MODEL = "google/byt5-large"  # 1.2B params, byte-level

# Check for pre-uploaded base model
for p in ["/kaggle/input/byt5-large-pretrained/", "/kaggle/input/google-byt5-large/"]:
    if Path(p).exists() and (Path(p) / 'config.json').exists():
        BASE_MODEL = p
        print(f"Found pre-uploaded base model: {p}")
        break

if not TRAIN_FROM_SCRATCH:
    for p in [
        "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
        "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
        "/kaggle/input/byt5-akkadian-mbr-v2",
    ]:
        if Path(p).exists():
            BASE_MODEL = p
            print(f"Using pre-trained: {p}")
            break

# --- Output ---
OUTPUT_DIR = Path("/kaggle/working/model") if Path("/kaggle").exists() else Path("../models/byt5-scratch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Sequence Lengths (ByT5 = byte-level) ---
MAX_SRC_LEN = 512
MAX_TGT_LEN = 512
PREFIX = "translate Akkadian to English: "
SEED = 42
MAX_BYTE_FILTER = 1024
NUM_EVAL_SAMPLES = 100

# --- Batch Config ---
if torch.cuda.is_available() and (IS_H100 or IS_A100):
    BATCH_SIZE, GRAD_ACCUM = 4, 8
    USE_BF16, USE_FP16 = True, False
elif torch.cuda.is_available():
    BATCH_SIZE, GRAD_ACCUM = 2, 16
    USE_BF16, USE_FP16 = False, True
else:
    BATCH_SIZE, GRAD_ACCUM = 1, 32
    USE_BF16, USE_FP16 = False, False

# --- 3-Phase Curriculum ---
if TRAIN_FROM_SCRATCH:
    PHASES = [
        {
            "name": "Phase 1: Broad (All Data)",
            "epochs": 3, "lr": 3e-4, "warmup_ratio": 0.05,
            "label_smoothing": 0.1, "weight_decay": 0.01,
            "sources": None,
        },
        {
            "name": "Phase 2: OA Domain (Sentence Pairs)",
            "epochs": 5, "lr": 5e-5, "warmup_ratio": 0.05,
            "label_smoothing": 0.05, "weight_decay": 0.01,
            "sources": ["strategy_a", "strategy_b_short", "strategy_b_aligned",
                        "strategy_b_doc", "strategy_c"],
        },
        {
            "name": "Phase 3: High-Quality Polish",
            "epochs": 3, "lr": 1e-5, "warmup_ratio": 0.1,
            "label_smoothing": 0.0, "weight_decay": 0.005,
            "sources": ["strategy_a"],
        },
    ]
else:
    PHASES = [
        {"name": "Phase 1: High-Quality Only", "epochs": 1, "lr": 5e-7,
         "warmup_ratio": 0.1, "label_smoothing": 0.0, "weight_decay": 0.01,
         "sources": ["strategy_a"]},
        {"name": "Phase 2: Broader Data", "epochs": 1, "lr": 2e-7,
         "warmup_ratio": 0.1, "label_smoothing": 0.0, "weight_decay": 0.01,
         "sources": ["strategy_a", "strategy_b_aligned", "strategy_c"]},
    ]

print(f"Mode: {'SCRATCH' if TRAIN_FROM_SCRATCH else 'FINE-TUNE'}")
print(f"Model: {BASE_MODEL}")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"Precision: {'BF16' if USE_BF16 else 'FP16' if USE_FP16 else 'FP32'}")
for i, p in enumerate(PHASES):
    print(f"  Phase {i+1}: {p['name']} | {p['epochs']}ep | lr={p['lr']}")

In [ ]:
# ============================================================
# V3 PREPROCESSING — matches competition test data format
# ============================================================
# Key rules from discussion:
# - ḫ/Ḫ → h/H (+0.4 LB)
# - <big_gap> → <gap> (v3 unified)
# - Deduplicate sequential gaps
# - PN → <gap> (Veenhof AKT 8)
# - Remove annotations: (fem), (plur), (sing)
# - Fractions → Unicode: ½ ⅓ ⅔ ¼ ¾ ⅙ ⅚
# - Remove orphan brackets, standardize quotes

def v3_normalize_transliteration(text):
    """Normalize transliteration to match v3 test format."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    # ḫ/Ḫ → h/H (test set uses h/H only)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    # Gap unification: everything → <gap>
    text = text.replace('<big_gap>', '<gap>')
    text = re.sub(r'\.{3,}|…+', '<gap>', text)
    text = re.sub(r'(?<![a-zA-Z\-])\bx\b(?![a-zA-Z\-])', '<gap>', text)
    # Deduplicate sequential gaps
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    # Remove orphan brackets (protect gaps)
    text = text.replace('<gap>', '\x00G\x00')
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'<<[^>]*>>', '', text)
    text = text.replace('\x00G\x00', '<gap>')
    # Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def v3_normalize_translation(text):
    """Normalize translation to match v3 test format."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    # PN → <gap> (Veenhof AKT 8 convention)
    text = re.sub(r'\bPN\b', '<gap>', text)
    # ḫ/Ḫ → h/H
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    # Gap unification
    text = text.replace('<big_gap>', '<gap>')
    text = re.sub(r'\.{3,}|…+', '<gap>', text)
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    # Remove annotations: (fem), (plur), (sing), (plural), (?), (!)
    text = re.sub(r'\s*\((fem|plur|pl|sing|singular|plural)\.?\s*\w*\)', '', text, flags=re.I)
    text = re.sub(r'\s*\([?!]\)', '', text)
    # Fractions → Unicode (BEFORE any char removal!)
    for frac, uni in {'1/3':'⅓','2/3':'⅔','1/6':'⅙','5/6':'⅚','1/2':'½','1/4':'¼','3/4':'¾'}.items():
        text = text.replace(frac, uni)
    text = re.sub(r'\b0\.5\b', '½', text)
    text = re.sub(r'(\d+)\.5\b', lambda m: f"{m.group(1)}½", text)
    text = re.sub(r'\b0\.25\b', '¼', text)
    text = re.sub(r'\b0\.75\b', '¾', text)
    text = re.sub(r'\b0\.333+\b', '⅓', text)
    text = re.sub(r'\b0\.666+\b', '⅔', text)
    # Remove brackets (protect gaps)
    text = text.replace('<gap>', '\x00G\x00')
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'<<[^>]*>>', '', text)
    text = text.replace('\x00G\x00', '<gap>')
    # Standardize quotes to straight
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201e', '"')
    # Gap spacing
    text = re.sub(r'(?<![- ])<gap>(?![- ])', ' <gap> ', text)
    # Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Self-test
_t1 = v3_normalize_transliteration("um-ma Ḫa-nu-um-ma [x] ... a-na")
assert 'ḫ' not in _t1.lower() and '<big_gap>' not in _t1, f"Failed: {_t1}"
_t2 = v3_normalize_translation("Seal of PN son of 1/3 mina (plur)")
assert 'PN' not in _t2 and '⅓' in _t2 and '(plur)' not in _t2, f"Failed: {_t2}"
print("V3 Preprocessing OK")
print(f"  Translit: '{_t1}'")
print(f"  Translat: '{_t2}'")

In [ ]:
# ============================================================
# LOAD & PREPROCESS DATA
# ============================================================
print(f"Loading: {TRAIN_FILE}")
train_data = pd.read_csv(TRAIN_FILE).dropna(subset=['transliteration', 'translation'])
val_data = pd.read_csv(VAL_FILE).dropna(subset=['transliteration', 'translation'])
print(f"Raw: {len(train_data)} train | {len(val_data)} val")

if 'source' in train_data.columns:
    print("\nSources:")
    print(train_data['source'].value_counts().to_string())

print("\nApplying v3 normalization...")
train_data['transliteration'] = train_data['transliteration'].apply(v3_normalize_transliteration)
train_data['translation'] = train_data['translation'].apply(v3_normalize_translation)
val_data['transliteration'] = val_data['transliteration'].apply(v3_normalize_transliteration)
val_data['translation'] = val_data['translation'].apply(v3_normalize_translation)

train_data = train_data[train_data['transliteration'].str.len() > 5]
train_data = train_data[train_data['translation'].str.len() > 5]
train_data = train_data.reset_index(drop=True)

src_bytes = train_data['transliteration'].str.encode('utf-8').str.len()
tgt_bytes = train_data['translation'].str.encode('utf-8').str.len()
mask = (src_bytes <= MAX_BYTE_FILTER) & (tgt_bytes <= MAX_BYTE_FILTER)
n_before = len(train_data)
train_data = train_data[mask].reset_index(drop=True)
print(f"After filter: {len(train_data)} train (dropped {n_before - len(train_data)})")

print("\nSample pairs:")
for i in range(min(3, len(train_data))):
    r = train_data.iloc[i]
    print(f"  SRC: {r['transliteration'][:80]}")
    print(f"  TGT: {r['translation'][:80]}\n")
print(f"Elapsed: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# DATASET & EVALUATION
# ============================================================

class Seq2SeqDataset(Dataset):
    def __init__(self, df, tokenizer, max_src=512, max_tgt=512, prefix=""):
        self.samples = []
        t0 = time.time()
        for _, row in df.iterrows():
            src = prefix + str(row['transliteration'])
            tgt = str(row['translation'])
            src_enc = tokenizer(src, max_length=max_src, truncation=True)
            tgt_enc = tokenizer(text_target=tgt, max_length=max_tgt, truncation=True)
            self.samples.append({
                'input_ids': src_enc['input_ids'],
                'attention_mask': src_enc['attention_mask'],
                'labels': tgt_enc['input_ids'],
            })
        print(f"  Tokenized {len(self.samples)} in {time.time()-t0:.1f}s")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def filter_by_source(data, sources):
    if sources is None:
        return data
    if 'source' not in data.columns:
        return data
    filtered = data[data['source'].isin(sources)]
    print(f"  Filtered: {len(filtered)} rows from {sources}")
    return filtered

def evaluate_model(model, tokenizer, val_ds, max_samples=100, num_beams=5):
    model.eval()
    device = next(model.parameters()).device
    preds, refs = [], []
    n = min(max_samples, len(val_ds))
    print(f"  Evaluating {n} samples...", end=" ", flush=True)
    t0 = time.time()
    for i in range(n):
        s = val_ds[i]
        ids = torch.tensor([s['input_ids']], device=device)
        mask = torch.tensor([s['attention_mask']], device=device)
        with torch.no_grad():
            out = model.generate(
                input_ids=ids, attention_mask=mask,
                max_new_tokens=256, num_beams=num_beams,
                length_penalty=1.3, repetition_penalty=1.2,
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        lab = [t for t in s['labels'] if t >= 0 and t != tokenizer.pad_token_id]
        lab = [min(t, 258) for t in lab]
        ref = tokenizer.decode(lab, skip_special_tokens=True).strip()
        preds.append(pred)
        refs.append(ref)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    geo = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0
    print(f"done in {time.time()-t0:.0f}s")
    print(f"  BLEU={bleu:.2f} | chrF++={chrf:.2f} | GeoMean={geo:.2f}")
    for j in range(min(3, n)):
        print(f"  [{j}] pred: {preds[j][:80]}")
        print(f"       ref:  {refs[j][:80]}")
    return {'bleu': round(bleu, 2), 'chrf_pp': round(chrf, 2), 'geo_mean': round(geo, 2)}

print("Dataset and evaluation defined.")

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================
print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
)
if torch.cuda.is_available():
    model = model.cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.0f}M | dtype: {next(model.parameters()).dtype}")

print("\nTokenizing validation set...")
val_ds = Seq2SeqDataset(val_data, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX)

if not TRAIN_FROM_SCRATCH:
    print("\nBaseline evaluation (before training):")
    baseline = evaluate_model(model, tokenizer, val_ds, NUM_EVAL_SAMPLES, num_beams=5)
    print(f"Baseline GeoMean = {baseline['geo_mean']}")
else:
    baseline = None

print(f"\nElapsed: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# 3-PHASE CURRICULUM TRAINING
# ============================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, label_pad_token_id=-100
)

all_results = {}
best_geo = baseline['geo_mean'] if baseline else 0.0
best_phase = 'baseline' if baseline else None
best_state = copy.deepcopy(model.state_dict()) if baseline else None

for phase_idx, cfg in enumerate(PHASES):
    phase_num = phase_idx + 1
    phase_name = cfg['name']

    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")

    phase_data = filter_by_source(train_data, cfg['sources'])
    if len(phase_data) < 10:
        print(f"Skipping - only {len(phase_data)} samples")
        continue

    print(f"Data: {len(phase_data)} | Epochs: {cfg['epochs']} | LR: {cfg['lr']}")
    train_ds = Seq2SeqDataset(phase_data, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX)

    total_steps = max(1, len(phase_data) * cfg['epochs'] // (BATCH_SIZE * GRAD_ACCUM))
    warmup_steps = int(total_steps * cfg['warmup_ratio'])
    log_steps = max(10, total_steps // 20)
    phase_dir = OUTPUT_DIR / f"phase{phase_num}"

    args = Seq2SeqTrainingArguments(
        output_dir=str(phase_dir),
        num_train_epochs=cfg['epochs'],
        learning_rate=cfg['lr'],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        weight_decay=cfg['weight_decay'],
        label_smoothing_factor=cfg['label_smoothing'],
        bf16=USE_BF16, fp16=USE_FP16,
        predict_with_generate=False,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=log_steps,
        report_to='none',
        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )

    print(f"Steps: ~{total_steps} | Warmup: {warmup_steps}")
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
    )

    trainer.train()

    print(f"\nEvaluating Phase {phase_num}...")
    results = evaluate_model(model, tokenizer, val_ds, NUM_EVAL_SAMPLES, num_beams=5)
    all_results[phase_name] = results

    if results['geo_mean'] > best_geo:
        best_geo = results['geo_mean']
        best_phase = phase_name
        best_state = copy.deepcopy(model.state_dict())
        best_dir = OUTPUT_DIR / 'best'
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(best_dir), safe_serialization=True)
        tokenizer.save_pretrained(str(best_dir))
        print(f"  New best! GeoMean={best_geo:.2f}")
    else:
        print(f"  No improvement (best: {best_geo:.2f} from {best_phase})")
        if not TRAIN_FROM_SCRATCH:
            print(f"  Restoring best model...")
            model.load_state_dict(best_state)
            break

    del trainer, train_ds
    gc.collect()
    torch.cuda.empty_cache()
    if phase_dir.exists():
        shutil.rmtree(phase_dir, ignore_errors=True)
    print(f"Phase {phase_num} done. Elapsed: {(time.time()-start_time)/60:.1f} min")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best: {best_phase} (GeoMean={best_geo:.2f})")
print(f"Total: {(time.time()-start_time)/60:.1f} min")
print(f"{'='*60}")

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

if best_state is not None:
    print(f"Restoring best model from {best_phase}...")
    model.load_state_dict(best_state)

model.save_pretrained(str(final_dir), safe_serialization=True)
tokenizer.save_pretrained(str(final_dir))

with open(str(final_dir / 'training_results.json'), 'w') as f:
    json.dump({
        'results': all_results,
        'best_phase': best_phase,
        'best_geo_mean': best_geo,
        'train_from_scratch': TRAIN_FROM_SCRATCH,
        'base_model': BASE_MODEL,
    }, f, indent=2)

print(f"Final model saved to: {final_dir}")
print(f"\nAll Results:")
for name, r in all_results.items():
    m = " <- BEST" if name == best_phase else ""
    print(f"  {name}: Geo={r['geo_mean']} | BLEU={r['bleu']} | chrF++={r['chrf_pp']}{m}")

model_size = sum(f.stat().st_size for f in final_dir.rglob('*') if f.is_file()) / 1e9
print(f"\nModel size: {model_size:.2f} GB")
print(f"Total: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# QUICK TEST
# ============================================================
model.eval()
device = next(model.parameters()).device

test_inputs = [
    "um-ma ša-lim-a-šùr-ma a-na šu-be-lim-ma",
    "1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé",
    "KIŠIB ma-nu-ba-lúm-a-šùr DUMU ṣí-lá-{d}IM",
    "a-na a-lim{ki} i-lá-ak-ma KÙ.BABBAR ú-šé-bi₄-lam",
    "ša-bu-ni-tum a-na pu-šu-ki-in qí-bi-ma",
]

print("Sample translations:\n")
for t in test_inputs:
    t_norm = v3_normalize_transliteration(t)
    enc = tokenizer(
        PREFIX + t_norm, max_length=MAX_SRC_LEN,
        truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, num_beams=5, length_penalty=1.3)
    trans = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  AKK: {t}")
    print(f"  ENG: {trans}\n")

print("Done! Upload model to Kaggle Models, then use byte-akkadian.ipynb for ensemble inference.")